In [ ]:
import numpy as np
from pysr import PySRRegressor

In [ ]:
# Load data
data = np.loadtxt("synthetic_Hz_data.csv", delimiter=",", skiprows=1)
X = data[:, :4]  # Features: H0, Omega_m0, Omega_Lambda0, z
Y = data[:, 4]   # Target: H

feature_names = ['H0', 'Omega_m0', 'Omega_Lambda0', 'z']

# Set up symbolic regression
# We adjust the operators for symbolic regression based on the anticipated complexity of the form
model = PySRRegressor(
    niterations=100,                  # Number of iterations of the search (increase for complex forms)
    populations=30,                  # Number of populations in each iteration
    #ncycles_per_iteration=,         # Number of algorithm cycles per iteration
    binary_operators=["+", "*", "/"],  # Typical arithmetic operators
    unary_operators=["sqrt"],  # Consider including logarithmic and exponential functions
    model_selection="best"           # Select the best model based on score
)

# Fit model
model.fit(X, Y, variable_names=feature_names)

# Print full Pareto front and best sympy expression
print(model)
best_expr = model.sympy()
print(f"Best sympy expression:",best_expr)

# For even more detailed analysis you can print top ranked equations
for equation in model.equations_:
    print(equation)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data = np.loadtxt("synthetic_Hz_data.csv", delimiter=",", skiprows=1)
H0 = data[:, 0]
Omega_m0 = data[:, 1]
Omega_Lambda0 = data[:, 2]
z = data[:, 3]
H_true = data[:, 4]

# Define the derived model from symbolic regression
def predicted_H_model(z, H0, Omega_Lambda0):
    return H0*z*(Omega_m0 + 0.04156827)*(Omega_m0*(2.2516038 - 2.96637*np.sqrt(z)) + 2.4461758*np.sqrt(z)) + H0 

# Compute model predictions
H_predicted = predicted_H_model(z, H0, Omega_Lambda0)

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(H_true, H_predicted, color='orange', alpha=0.5, label='Predicted vs Actual')
plt.title('Comparison of Symbolic Regression Predictions with Actual Data')
plt.xlabel('Actual H(z)')
plt.ylabel('Predicted H(z)')
plt.plot([H_true.min(), H_true.max()], [H_true.min(), H_true.max()], 'k--', label='Ideal Fit')  # Ideal fit line
plt.legend()
plt.grid(True)
plt.show()

# Optionally, calculate metrics such as RMSE
rmse = np.sqrt(np.mean((H_true - H_predicted)**2))
print(f"Root Mean Squared Error: {rmse:.2f}")


In [ ]:
# Calculate fractional error
frac_error = 100 * np.abs(H_true - H_predicted)/H_true

# Plotting residuals
plt.figure(figsize=(8, 4))
plt.scatter(z, frac_error, color='red', alpha=0.5)
#plt.title('Percentage fractional error')
plt.xlabel('Redshift (z)')
plt.ylabel('Percentage fractional error')
plt.axhline(0, color='black', linewidth=0.8)
plt.grid(True)
plt.show()
